# 10.4 - What Is MCP

**Module 10 - Tool Use & MCP** | Netsetos GenAI Engineering

This notebook works through What Is MCP hands-on: Why MCP exists: N x M becomes N + M; MCP messages: JSON-RPC 2.0 under the hood; Build a working MCP server.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## 1 - Why MCP exists: N x M becomes N + M

Before a single line of protocol code, it helps to see the problem MCP solves in pure arithmetic. Every app that wants to use a tool normally writes a custom integration in that app's own format, so the work grows as apps times tools. This cell counts both worlds side by side so the payoff is a number, not a slogan.

In [ ]:
# THE N x M PROBLEM - why one shared protocol is worth it.
apps  = ["Claude Desktop", "Cursor", "VS Code", "OpenAI", "Gemini"]
tools = ["weather", "database", "calendar", "github", "slack", "search"]
M, N = len(apps), len(tools)

bespoke  = M * N          # without MCP: every app wires every tool, in that app's own format
with_mcp = M + N          # with MCP: each app + each tool implements ONE protocol

print(f"{M} apps x {N} tools")
print(f"  without MCP: {M} x {N} = {bespoke} bespoke integrations")
print(f"  with MCP:    {M} + {N} = {with_mcp} MCP connections")
print(f"  saved:       {bespoke - with_mcp} integrations ({100*(bespoke-with_mcp)//bespoke}% fewer)")

# Output:
# 5 apps x 6 tools
#   without MCP: 5 x 6 = 30 bespoke integrations
#   with MCP:    5 + 6 = 11 MCP connections
#   saved:       19 integrations (63% fewer)

**Explanation**

This is a back-of-the-envelope calculation, not a model call - it just multiplies and adds two list lengths to contrast the integration cost with and without a shared protocol. The core idea: without a standard, every app-tool pair is bespoke (M x N); with MCP, each app and each tool implements the protocol once (M + N). The gap between those two numbers is the entire business case for MCP.

**How the code works - step by step**
- **`apps` and `tools`** - two plain lists standing in for AI hosts (Claude Desktop, Cursor, ...) and the tools they might want (weather, database, ...).
- **`M, N`** - the counts of each list, so the math is driven by real lengths rather than hard-coded numbers.
- **`bespoke = M * N`** - the no-protocol world: every app wires every tool in that app's own format.
- **`with_mcp = M + N`** - the MCP world: each app and each tool speaks one protocol, so they only connect to the protocol, not to each other.
- **the `print` block** - reports both totals and the percentage saved via integer math `100*(bespoke-with_mcp)//bespoke`.

**In one line:** count apps and tools -> compare M x N against M + N -> print how many integrations the protocol erases.

**What you'll see:** Four printed lines: "5 apps x 6 tools", then 30 bespoke integrations without MCP, 11 connections with MCP, and "saved: 19 integrations (63% fewer)" - matching the # Output: block at the bottom of the cell.

## 2 - MCP messages: JSON-RPC 2.0 under the hood

MCP is not a new wire format - it is JSON-RPC 2.0 with a small set of agreed method names. This cell builds the four messages that make up a full interaction as plain Python dicts so you can read the protocol directly instead of trusting an SDK to hide it.

In [ ]:
# MCP MESSAGE FORMAT — JSON-RPC 2.0
import json

# 1. Initialize: client discovers server
init = {"jsonrpc":"2.0", "id":1, "method":"initialize",
        "params": {"protocolVersion":"2025-03-26",
                   "clientInfo":{"name":"netsetos-app"}}}

# 2. List tools: what can this server do?
list_tools = {"jsonrpc":"2.0", "id":2, "method":"tools/list"}

# 3. Call tool: execute a function
call = {"jsonrpc":"2.0", "id":3, "method":"tools/call",
        "params": {"name":"get_weather", "arguments":{"city":"Hyderabad"}}}

# Response
result = {"jsonrpc":"2.0", "id":3,
          "result":{"content":[{"type":"text","text":"34°C, Sunny"}]}}

print("MCP Flow: initialize → tools/list → tools/call → result")
print("Protocol: JSON-RPC 2.0 over any transport")


**Explanation**

This cell is a set of hand-written message literals, not a live client - nothing is sent anywhere; the dicts exist so you can inspect the exact shape of each MCP call. The core idea is that the whole protocol is three request methods plus their responses: initialize (handshake), tools/list (discover), tools/call (execute), each tagged with a matching id so a response can be paired to its request.

**How the code works - step by step**
- **`init`** - the handshake: method `initialize`, carrying a `protocolVersion` and `clientInfo` so server and client agree on version and identity.
- **`list_tools`** - method `tools/list` with no params: the client asking what functions this server exposes.
- **`call`** - method `tools/call`, whose `params` name the tool (`get_weather`) and pass `arguments` (`city: Hyderabad`).
- **`result`** - the server's reply, keyed to the same `id: 3`, returning a `content` list of typed text blocks.
- **the `print` lines** - narrate the flow and the fact that this same JSON rides over any transport.

**In one line:** initialize -> tools/list -> tools/call -> result, all as JSON-RPC 2.0 dicts with matching ids.

**What you'll see:** Two printed lines: "MCP Flow: initialize -> tools/list -> tools/call -> result" and "Protocol: JSON-RPC 2.0 over any transport". The dicts themselves are defined but not printed.

## Setup - install the MCP SDK

The next block imports from the `mcp` package, which is not preinstalled in Colab. Run this once before the server cell.

In [ ]:
# Setup — install the MCP SDK (not preinstalled in Colab)
%pip install -q mcp

**Explanation**

A one-line environment prep cell that pip-installs the official MCP SDK quietly so the server imports below resolve.

**How the code works - step by step**
- **`%pip install -q mcp`** - installs the `mcp` package (the Python MCP SDK) with `-q` to suppress the install log.

**In one line:** get the `mcp` package onto the machine so the next cell can import `Server`, `Tool`, and `TextContent`.

**What you'll see:** (no output - environment prep)

## 3 - Build a working MCP server

Having seen the raw messages, this cell hides them behind the SDK and builds a real, if tiny, server. Two decorators register everything: one that advertises the available tools, and one that runs them when called. This is the server half of the host/client/server architecture that MCP is built on.

> **Needs the `mcp` package** (installed in the setup cell above).

In [ ]:
# MCP SERVER — Build a working server
# pip install mcp
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent

app = Server("netsetos-tools")

@app.list_tools()
async def list_tools():
    return [
        Tool(name="get_weather", description="Get weather for a city",
             inputSchema={"type":"object","properties":{"city":{"type":"string"}},"required":["city"]}),
        Tool(name="search_courses", description="Search Netsetos courses",
             inputSchema={"type":"object","properties":{"topic":{"type":"string"}},"required":["topic"]}),
    ]

@app.call_tool()
async def call_tool(name, arguments):
    if name == "get_weather":
        return [TextContent(type="text", text=f"34°C Sunny in {arguments['city']}")]
    elif name == "search_courses":
        return [TextContent(type="text", text="GenAI Complete: 14999 INR")]

print("MCP Server: @app.list_tools + @app.call_tool")
print("Run via stdio, add to claude_desktop_config.json")


**Explanation**

This is a server definition, not a running process - it registers handlers on a `Server` object but the cell only defines them, it does not start the stdio event loop. The core idea is that an MCP server is two async callbacks: `list_tools` answers the discovery request from Step 2, and `call_tool` answers the execution request, together turning the raw JSON-RPC methods into ordinary Python functions.

**How the code works - step by step**
- **`app = Server("netsetos-tools")`** - creates the server instance and names it.
- **`@app.list_tools()`** - registers the discovery handler; it returns a list of `Tool` objects, each with a name, description, and a JSON-Schema `inputSchema` declaring required arguments (`get_weather` needs `city`, `search_courses` needs `topic`).
- **`@app.call_tool()`** - registers the execution handler; it branches on the tool `name` and returns a `TextContent` block, reading values out of `arguments`.
- **the `print` lines** - confirm the two decorators and note that the server is meant to run over stdio and be wired into `claude_desktop_config.json`.

**In one line:** name a server, declare its tools with `@list_tools`, execute them with `@call_tool`.

**What you'll see:** Two printed lines: "MCP Server: @app.list_tools + @app.call_tool" and "Run via stdio, add to claude_desktop_config.json". The tool handlers are registered but not invoked here, so no weather or course result is printed.

Across these blocks you saw MCP from three angles: the economics (N+M beats N×M), the wire format (three JSON-RPC messages carry the entire protocol), and the code (a real server built from two decorators). The through-line is that MCP is a thin, shared contract — initialize, discover, call — that any host and any tool can speak, replacing a grid of bespoke integrations. Next up in Module 10, you'll wire a client to a server like this one and let a model actually drive the tools it exposes.